In [8]:
from typing import TypeAlias

Clue: TypeAlias = list[int]
RowClues: TypeAlias = list[Clue]
ColClues: TypeAlias = list[Clue]
NonogramQuestion: TypeAlias = tuple[RowClues, ColClues]
NonogramState: TypeAlias = list[list[bool | None]]  # True: filled, False: empty, None: unknown

In [9]:
import tkinter as tk
import threading
import queue


class NonogramDrawer:
    def __init__(self) -> None:
        self.cell_size = 50
        self.padding = 5
        self._command_queue: queue.Queue = queue.Queue()
        self._ready = threading.Event()
        self._closed = threading.Event()
        self._thread: threading.Thread | None = None
        self.window: tk.Tk | None = None
        self.canvas: tk.Canvas | None = None

    def run(self) -> None:
        if self._thread is not None and self._thread.is_alive():
            return
        self._ready.clear()
        self._closed.clear()
        self._command_queue = queue.Queue()
        self._thread = threading.Thread(target=self._ui_thread, daemon=True, name="NonogramDrawerUI")
        self._thread.start()
        if not self._ready.wait(timeout=2):
            raise RuntimeError("Failed to start Nonogram drawer UI thread")

    def draw(self, question: NonogramQuestion, state: NonogramState | None = None) -> None:
        row_clues, col_clues = question
        rows = len(row_clues)
        cols = len(col_clues)

        if state is not None:
            if len(state) != rows or any(len(row) != cols for row in state):
                raise ValueError("State dimensions do not match the question grid size")

        self.run()
        if not self._ready.wait(timeout=2):
            raise RuntimeError("Drawer UI thread is not ready")
        self._command_queue.put(("draw", (question, state)))

    def close(self) -> None:
        if self._thread is None:
            return
        if not self._closed.is_set():
            self._command_queue.put(("close", None))
            self._closed.wait(timeout=2)
        if self._thread.is_alive():
            self._thread.join(timeout=2)
        self._thread = None
        self._ready.clear()

    def _ui_thread(self) -> None:
        self.window = tk.Tk()
        self.window.title("Nonogram Drawer")
        self.window.resizable(False, False)
        self.window.protocol("WM_DELETE_WINDOW", self._handle_close_request)
        self.canvas = None
        self._ready.set()
        self._closed.clear()
        self._poll_commands()
        self.window.mainloop()
        self._cleanup_after_loop()

    def _poll_commands(self) -> None:
        if self.window is None or self._closed.is_set():
            return
        try:
            while True:
                command, payload = self._command_queue.get_nowait()
                if command == "draw" and payload is not None:
                    question, state = payload
                    self._render(question, state)
                elif command == "close":
                    self._handle_close_request()
                else:
                    continue
        except queue.Empty:
            pass
        finally:
            if self.window is not None and not self._closed.is_set():
                self.window.after(16, self._poll_commands)

    def _handle_close_request(self) -> None:
        if self._closed.is_set():
            return
        self._closed.set()
        if self.window is not None:
            try:
                self.window.quit()
            except tk.TclError:
                pass
            try:
                self.window.destroy()
            except tk.TclError:
                pass

    def _cleanup_after_loop(self) -> None:
        self.canvas = None
        self.window = None
        self._closed.set()
        self._ready.clear()
        try:
            while True:
                self._command_queue.get_nowait()
        except queue.Empty:
            pass

    def _render(self, question: NonogramQuestion, state: NonogramState | None) -> None:
        if self.window is None:
            return
        row_clues, col_clues = question
        rows = len(row_clues)
        cols = len(col_clues)

        max_row_clues = max((len(c) for c in row_clues), default=0)
        max_col_clues = max((len(c) for c in col_clues), default=0)

        left_margin = max_row_clues * self.cell_size + self.padding * 2
        top_margin = max_col_clues * self.cell_size + self.padding * 2

        width = left_margin + cols * self.cell_size + self.padding
        height = top_margin + rows * self.cell_size + self.padding

        if self.canvas is None:
            self.canvas = tk.Canvas(self.window, bg="white")
            self.canvas.pack()

        self.canvas.configure(width=width, height=height)
        self.canvas.delete("all")

        # Draw row clues (right-aligned beside each row)
        for i, clues in enumerate(row_clues):
            y = top_margin + i * self.cell_size + self.cell_size / 2
            for k, v in enumerate(reversed(clues)):
                x = left_margin - (k + 0.5) * self.cell_size
                self.canvas.create_text(x, y, text=str(v), font=("Arial", int(self.cell_size * 0.4)))

        # Draw column clues (bottom-aligned above each column)
        for j, clues in enumerate(col_clues):
            x = left_margin + j * self.cell_size + self.cell_size / 2
            for k, v in enumerate(reversed(clues)):
                y = top_margin - (k + 0.5) * self.cell_size
                self.canvas.create_text(x, y, text=str(v), font=("Arial", int(self.cell_size * 0.4)))

        # Draw grid cells and state overlay if provided
        for i in range(rows):
            for j in range(cols):
                x0 = left_margin + j * self.cell_size
                y0 = top_margin + i * self.cell_size
                x1 = x0 + self.cell_size
                y1 = y0 + self.cell_size

                cell_state = state[i][j] if state is not None else None

                fill_color = "white"
                if cell_state is True:
                    fill_color = "black"
                elif cell_state is None:
                    fill_color = "white"  # Placeholder for unknown; adjust if you want a grey tone

                self.canvas.create_rectangle(x0, y0, x1, y1, outline="black", fill=fill_color)

                if cell_state is False:
                    inset = self.cell_size * 0.2
                    self.canvas.create_line(x0 + inset, y0 + inset, x1 - inset, y1 - inset, fill="gray40", width=2)
                    self.canvas.create_line(x0 + inset, y1 - inset, x1 - inset, y0 + inset, fill="gray40", width=2)


In [ ]:
col_clues = [
    [0],
    [5],
    [2],
    [2],
    [0]
]
row_clues = [
    [3],
    [3],
    [1],
    [1],
    [1]
]
init_state: NonogramState = [[False, True, None, None, False]] * 5

In [ ]:
question: NonogramQuestion = (row_clues, col_clues)
drawer: NonogramDrawer = NonogramDrawer()
drawer.run()
drawer.draw(question, init_state)